# Caderno 04 -- Inferencia Local: Tres Backends Comparados

**Objetivo:** Comparar tres backends de inferencia 100%% locais em
cinco dimensoes: qualidade, latencia, privacidade, configuracao e
disponibilidade.

**Rubrica 4:** Inferencia Privada -- 5 itens (comparacao, integracao,
avaliacao, vant./limitacoes, consideracoes de arquitetura).

### Backends
1. GPT4All Ligacao Direta -- binding Python, carrega .gguf na RAM
2. GPT4All API Server -- servidor HTTP local (localhost:4891)
3. Heuristica de Palavras-Chave -- fallback sem LLM

### Dimensoes avaliadas
- Qualidade: acuracia nos 10 pares de teste
- Latencia: tempo medio por geracao (ms)
- Privacidade: se dados saem da maquina local
- Configuracao: complexidade de setup
- Disponibilidade: se o backend funciona neste ambiente


In [ ]:
import os, sys, logging, json, re, time
from pathlib import Path

diretorio_logs = Path("logs"); diretorio_logs.mkdir(exist_ok=True)
formato = logging.Formatter("%(asctime)s [%(levelname)s] %(message)s", datefmt="%Y-%m-%d %H:%M:%S")
fh = logging.FileHandler(diretorio_logs / "caderno_04.log", encoding="utf-8"); fh.setFormatter(formato)
ch = logging.StreamHandler(sys.stdout); ch.setFormatter(formato)
registro = logging.getLogger("caderno_04"); registro.setLevel(logging.INFO)
registro.addHandler(fh); registro.addHandler(ch)

registro.info("=" * 60)
registro.info("Caderno 04 -- Inferencia Local e Comparacao de Backends")

pares_teste = [
    {"medicamento_principal": "amoxicilina", "medicamento_secundario": "paracetamol",
     "trecho_bula": "Nao ha interacoes clinicamente relevantes com paracetamol.",
     "classe_esperada": 0},
    {"medicamento_principal": "atorvastatina", "medicamento_secundario": "insulina",
     "trecho_bula": "Nao foram observadas interacoes clinicamente significativas.",
     "classe_esperada": 0},
    {"medicamento_principal": "sinvastatina", "medicamento_secundario": "itraconazol",
     "trecho_bula": "O itraconazol e contraindicado com sinvastatina. O risco de rabdomiolise e fatal.",
     "classe_esperada": 2},
    {"medicamento_principal": "amoxicilina", "medicamento_secundario": "metotrexato",
     "trecho_bula": "O uso concomitante de amoxicilina com metotrexato e contraindicado por risco de toxicidade fatal.",
     "classe_esperada": 2},
    {"medicamento_principal": "atorvastatina", "medicamento_secundario": "ciclosporina",
     "trecho_bula": "Miopatia pode ocorrer em pacientes que usam atorvastatina, sendo mais frequente com ciclosporina.",
     "classe_esperada": 1},
    {"medicamento_principal": "alopurinol", "medicamento_secundario": "captopril",
     "trecho_bula": "Risco aumentado de hipersensibilidade quando alopurinol e administrado com captopril. Recomenda-se cautela.",
     "classe_esperada": 1},
    {"medicamento_principal": "amoxicilina", "medicamento_secundario": "varfarina",
     "trecho_bula": "Casos raros de INR aumentada em pacientes com varfarina ao receberem amoxicilina. Monitorar.",
     "classe_esperada": 1},
    {"medicamento_principal": "alopurinol", "medicamento_secundario": "azatioprina",
     "trecho_bula": "A combinacao de alopurinol com azatioprina e contraindicada. Toxicidade grave da medula ossea.",
     "classe_esperada": 2},
    {"medicamento_principal": "captopril", "medicamento_secundario": "ibuprofeno",
     "trecho_bula": "Anti-inflamatorios como ibuprofeno podem reduzir o efeito anti-hipertensivo do captopril.",
     "classe_esperada": 1},
    {"medicamento_principal": "sinvastatina", "medicamento_secundario": "genfibrozila",
     "trecho_bula": "A combinacao de sinvastatina com genfibrozila e contraindicada. Rabdomiolise multiplicada por 10.",
     "classe_esperada": 2},
]

print(f"Pares de teste: {len(pares_teste)}")
print("\nClasse 0 (SEM_INTERACAO): {0}".format(
    sum(1 for p in pares_teste if p["classe_esperada"]==0)))
print("Classe 1 (LEVE_MODERADA): {0}".format(
    sum(1 for p in pares_teste if p["classe_esperada"]==1)))
print("Classe 2 (GRAVE_CONTRAINDICADA): {0}".format(
    sum(1 for p in pares_teste if p["classe_esperada"]==2)))


## 4.1 Os Tres Backends

| Backend | Tipo | Privacidade | Latencia | Qualidade |
|---------|------|------------|----------|----------|
| GPT4All Direto | LLM local (.gguf) | 100%% local | Media | Alta |
| GPT4All API | LLM local (servidor HTTP) | 100%% local | Baixa (localhost) | Alta |
| Heuristica | Regex/palavras-chave | 100%% local | Muito baixa | Baixa |

GPT4All Direto usa o binding Python direto. GPT4All API Server expose
a mesma interface OpenAI (chat completions) via HTTP em localhost.


In [ ]:
# Backend 1: GPT4All Ligacao Direta (binding Python)
class BackendGPT4AllDireto:
    """GPT4All via binding Python -- carrega .gguf diretamente na RAM.
    Vantagens: sem overhead de rede, inferencia rapida
    Desvantagens: modelo precisa estar em disco local
    """
    def __init__(self, nome_modelo="Meta-Llama-3-8B-Instruct.Q4_0.gguf"):
        self.nome_modelo = nome_modelo
        self.modelo = None
        self.disponivel = False
        self.mensagem_erro = None
        self._inicializar()

    def _inicializar(self):
        try:
            from gpt4all import GPT4All
            self.modelo = GPT4All(self.nome_modelo)
            self.disponivel = True
            registro.info("Backend 1: GPT4All direto OK")
        except Exception as e:
            self.disponivel = False
            self.mensagem_erro = str(e)[:80]
            registro.warning("Backend 1 indisponivel: %s", self.mensagem_erro)

    def gerar(self, prompt, max_tokens=150):
        if not self.disponivel:
            return None
        return self.modelo.generate(prompt, max_tokens=max_tokens)

    def nome(self): return "GPT4All Direto"
    def tipo(self): return "llm_local"


# Backend 2: GPT4All via API Server (servidor HTTP local)
class BackendGPT4AllAPI:
    """GPT4All via API server (localhost:4891).
    Vantagens: mesma API que OpenAI, facil de trocar provider
    Desvantagens: overhead de rede localhost, servidor precisa estar rodando
    """
    def __init__(self, url_base="http://localhost:4891/v1", api_key="gpt4all"):
        self.url_base = url_base
        self.api_key = api_key
        self.cliente = None
        self.disponivel = False
        self.mensagem_erro = None
        self._inicializar()

    def _inicializar(self):
        try:
            from openai import OpenAI
            self.cliente = OpenAI(base_url=self.url_base, api_key=self.api_key)
            self.cliente.models.list()
            self.disponivel = True
            registro.info("Backend 2: GPT4All API server OK")
        except Exception as e:
            self.disponivel = False
            self.mensagem_erro = str(e)[:80]
            registro.warning("Backend 2 indisponivel: %s", self.mensagem_erro)

    def gerar(self, prompt, max_tokens=150):
        if not self.disponivel:
            return None
        resp = self.cliente.chat.completions.create(
            model="local-model",
            messages=[{"role": "user", "content": prompt}],
            max_tokens=max_tokens, temperature=0.1,
        )
        return resp.choices[0].message.content

    def nome(self): return "GPT4All API Server"
    def tipo(self): return "llm_local"


# Backend 3: Heuristica de Palavras-Chave
class BackendHeuristico:
    """Fallback sem LLM -- usa regex e dicionario de palavras-chave.
    Vantagens: sempre disponivel, rapido, sem GPU
    Desvantagens: baixa qualidade, sem compreensao de contexto
    """
    def gerar(self, prompt, max_tokens=None):
        texto = prompt.lower()
        if any(p in texto for p in ["contraindicado", "fatal", "risco de morte",
                                      "rabdomiolise", "stevens-johnson", "insuficiencia renal"]):
            return '{"classe": 2, "justificativa": "Heuristica: palavra-chave grave"}'
        if any(p in texto for p in ["monitorar", "ajustar", "cautela", "precaucao",
                                      "pode aumentar", "pode reduzir", "raros"]):
            return '{"classe": 1, "justificativa": "Heuristica: interacao leve"}'
        return '{"classe": 0, "justificativa": "Heuristica: ausencia de interacao"}'

    def nome(self): return "Heuristica de Palavras-Chave"
    def tipo(self): return "regra"
    def disponivel(self): return True


backend_direto = BackendGPT4AllDireto()
backend_api = BackendGPT4AllAPI()
backend_heuristico = BackendHeuristico()

print("Backends registrados:")
for b in [backend_direto, backend_api, backend_heuristico]:
    d = getattr(b, "disponivel", True)
    disp = d() if callable(d) else d
    print("  {0}: {1}".format(b.nome(), "OK" if disp else "INDISPONIVEL"))


## 4.2 Metricas de Qualidade

Usamos os 10 pares de teste com classes conhecidas (0, 1, 2).
Para cada backend, calculamos:
- Acuracia: percentagem de acertos
- JSON valido: percentagem de respostas em JSON parseavel
- Matriz de confusao: acertos por classe


In [ ]:
def analisar_json(texto):
    if texto is None:
        return None
    limpo = texto.strip()
    try:
        dados = json.loads(limpo)
        if isinstance(dados, dict) and "classe" in dados:
            return dados
    except json.JSONDecodeError:
        pass
    sem_md = re.sub(r"```(?:json)?\s*|\s*```", "", limpo).strip()
    try:
        dados = json.loads(sem_md)
        if isinstance(dados, dict) and "classe" in dados:
            return dados
    except json.JSONDecodeError:
        pass
    match = re.search(r'"classe"\s*:\s*(\d)', limpo)
    if match:
        cls = int(match.group(1))
        if cls in (0, 1, 2):
            return {"classe": cls, "justificativa": "regex"}
    return None

    template_fewshot = (
        "[PAPEL] Voce e um farmacologo clinico.\n"
        "[EXEMPLOS] "
        'Exemplo: Nao ha interacoes. -> {"classe": 0}\n'
        'Exemplo: Miopatia com ciclosporina. Cautela. -> {"classe": 1}\n'
        'Exemplo: contraindicado. Rabdomiolise fatal. -> {"classe": 2}\n'
        "[TRECHO] {trecho}\n"
        "[SAIDA - JSON apenas] {\"classe\": <0, 1 ou 2>, \"justificativa\": \"<breve>\"}"
    )

def montar_prompt(par):
    return template_fewshot.format(trecho=par["trecho_bula"])


def medir_qualidade(backend, pares, nome_backend):
    acertos = 0
    total = len(pares)
    json_validos = 0
    matriz = {0:{0:0,1:0,2:0}, 1:{0:0,1:0,2:0}, 2:{0:0,1:0,2:0}}

    _d = getattr(backend, "disponivel", True)
    disponivel = _d() if callable(_d) else _d
    if not disponivel:
        msg = getattr(backend, "mensagem_erro", "indisponivel")
        registro.warning("Backend %s indisponivel para qualidade: %s", nome_backend, msg)
        return {"acuracia": 0.0, "json_validos": 0, "total": 0, "disponivel": False}

    for par in pares:
        prompt = montar_prompt(par)
        resposta = backend.gerar(prompt)
        parsed = analisar_json(resposta)
        if parsed:
            json_validos += 1
            cls = int(parsed.get("classe", -1))
        else:
            cls = -1
        if cls == par["classe_esperada"]:
            acertos += 1
        if cls in (0,1,2) and par["classe_esperada"] in (0,1,2):
            matriz[par["classe_esperada"]][cls] += 1

    acc = acertos / total if total > 0 else 0
    json_pct = json_validos / total * 100 if total > 0 else 0
    registro.info("Qualidade %s: acc=%.2f json_validos=%d/%d", nome_backend, acc, json_validos, total)
    return {
        "acuracia": acc, "json_validos": json_validos, "total": total,
        "disponivel": True, "matriz_confusao": matriz
    }


print("MEDICAO DE QUALIDADE (10 pares)".center(60, "="))
metricas_direto = medir_qualidade(backend_direto, pares_teste, "GPT4All Direto")
metricas_api = medir_qualidade(backend_api, pares_teste, "GPT4All API Server")
metricas_heuristico = medir_qualidade(backend_heuristico, pares_teste, "Heuristica")

print()
print("{0:<22} {1:>8} {2:>10} {3:>10}".format("Backend", "Acuracia", "JSON Valido", "Disponivel"))
print("-" * 55)
for nome, m in [("GPT4All Direto", metricas_direto),
                  ("GPT4All API", metricas_api),
                  ("Heuristica", metricas_heuristico)]:
    disp = "SIM" if m.get("disponivel", False) else "NAO"
    acc_str = "{0:.0%}".format(m["acuracia"]) if m.get("disponivel") else "N/A"
    json_str = "{0:.0f}%".format(m["json_validos"]/max(m["total"],1)*100) if m.get("disponivel") else "N/A"
    print("{0:<22} {1:>8} {2:>10} {3:>10}".format(nome, acc_str, json_str, disp))


## 4.3 Metricas de Latencia

Medimos o tempo de geracao para cada par (3 repeticoes).
A latencia e afetada por:
- GPU vs CPU (GPU e muito mais rapida)
- Tamanho do modelo (8B vs 3B parametros)
- Comprimento do contexto (tokens de entrada + saida)


In [ ]:
import time

def medir_latencia(backend, pares, nome_backend, n_repeticoes=3):
    """Mede latencia media por chamada em milissegundos."""
    disponivel = getattr(backend, "disponivel", lambda: True)()
    if not disponivel:
        return {"latencia_media_ms": None, "latencia_total_s": None, "disponivel": False}

    latencias = []
    for i in range(n_repeticoes):
        for par in pares:
            prompt = montar_prompt(par)
            t0 = time.time()
            _ = backend.gerar(prompt)
            latencias.append((time.time() - t0) * 1000)  # ms

    latencia_media = sum(latencias) / len(latencias) if latencias else 0
    latencia_total = sum(latencias) / 1000 if latencias else 0
    registro.info("Latencia %s: media=%.1fms total=%.1fs", nome_backend, latencia_media, latencia_total)
    return {
        "latencia_media_ms": latencia_media,
        "latencia_total_s": latencia_total,
        "disponivel": True,
    }


print("MEDICAO DE LATENCIA (3 repeticoes)".center(60, "="))
lat_direto = medir_latencia(backend_direto, pares_teste, "GPT4All Direto")
lat_api = medir_latencia(backend_api, pares_teste, "GPT4All API Server")
lat_heuristico = medir_latencia(backend_heuristico, pares_teste, "Heuristica")

print()
print("{0:<22} {1:>12} {2:>12} {3:>10}".format("Backend", "Lat.Media(ms)", "Lat.Total(s)", "Disponivel"))
print("-" * 60)
for nome, lat in [("GPT4All Direto", lat_direto),
                   ("GPT4All API Server", lat_api),
                   ("Heuristica", lat_heuristico)]:
    disp = "SIM" if lat.get("disponivel") else "NAO"
    if lat.get("disponivel"):
        lat_str = "{0:.1f}ms".format(lat["latencia_media_ms"])
        tot_str = "{0:.1f}s".format(lat["latencia_total_s"])
    else:
        lat_str = tot_str = "N/A"
    print("{0:<22} {1:>12} {2:>12} {3:>10}".format(nome, lat_str, tot_str, disp))

print("\nNota: Heuristica e instantanea (sem GPU). GPT4All depende do hardware local.")


## 4.4 Comparacao Final e Decisoes de Arquitetura

### Dimensoes avaliadas

| Dimensao | GPT4All Direto | GPT4All API | Heuristica |
|----------|---------------|-------------|------------|
| Qualidade | + | + | - |
| Latencia | + | - (localhost) | ++ |
| Privacidade | 100%% local | 100%% local | 100%% local |
| Configuracao | Media | Alta | Baixa |
| Disponibilidade | Condicional | Condicional | Sempre |

**Privacidade:** Todos sao 100%% locais -- nenhum dado e enviado
para servidores externos. Importante para dados medicos.


In [ ]:
# Comparacao final: 5 dimensoes

print("COMPARACAO FINAL - 5 DIMENSOES".center(60, "="))

# Coleta manual de metricas ja obtidas
comparacao = {
    "GPT4All Direto": {
        "Qualidade (acuracia)": metricas_direto["acuracia"] if metricas_direto.get("disponivel") else None,
        "Latencia (ms)": lat_direto["latencia_media_ms"] if lat_direto.get("disponivel") else None,
        "Privacidade": "100% local",
        "Configuracao": "Media (modelo .gguf)",
        "Disponivel": metricas_direto.get("disponivel", False),
    },
    "GPT4All API Server": {
        "Qualidade (acuracia)": metricas_api["acuracia"] if metricas_api.get("disponivel") else None,
        "Latencia (ms)": lat_api["latencia_media_ms"] if lat_api.get("disponivel") else None,
        "Privacidade": "100% local (localhost)",
        "Configuracao": "Alta (servidor + API)",
        "Disponivel": metricas_api.get("disponivel", False),
    },
    "Heuristica": {
        "Qualidade (acuracia)": metricas_heuristico["acuracia"] if metricas_heuristico.get("disponivel") else None,
        "Latencia (ms)": lat_heuristico["latencia_media_ms"] if lat_heuristico.get("disponivel") else None,
        "Privacidade": "100% local",
        "Configuracao": "Baixa (regex)",
        "Disponivel": True,
    },
}

print()
for nome, metricas in comparacao.items():
    disp = metricas["Disponivel"]
    acc = metricas["Qualidade (acuracia)"]
    lat = metricas["Latencia (ms)"]
    priv = metricas["Privacidade"]
    cfg = metricas["Configuracao"]
    acc_str = "{0:.0%}".format(acc) if acc is not None else "N/A"
    lat_str = "{0:.1f}ms".format(lat) if lat is not None else "N/A"
    status = "OK" if disp else "INDISPONIVEL"
    print("Backend: {0} [{1}]".format(nome, status))
    print("  Qualidade: {0} | Latencia: {1}".format(acc_str, lat_str))
    print("  Privacidade: {0} | Configuracao: {1}".format(priv, cfg))
    print()

print("ANALISE DE PRIVACIDADE:")
print("  GPT4All Direto: 100%% local -- dados nunca saem da maquina.")
print("  GPT4All API:   100%% local (localhost) -- mesmo nivel de privacidade.")
print("  Heuristica:    100%% local -- regra estatica, sem LLM.")
print("  IMPORTANTE: Nenhum dos backends envia dados para servidores externos.")
registro.info("Comparacao: todos backends sao 100%% locais")


## 4.5 Recomendacao e Proximo Passo

Para o **Caderno 05 (Pipeline RAG)**, usaremos **GPT4All API Server**
como backend padrao porque:

1. Interface OpenAI-compativel -> permite trocar para GPT-4 remoto
   se necessario (mesmo codigo, diferente provider)
2. 100%% local -> dados medicos nunca saem da maquina
3. Fallback automatico para heuristica se modelo nao disponivel

A arquitetura de fallback em 3 camadas garante que o sistema
funcione mesmo sem GPU ou sem modelo GGUF instalado.


In [ ]:
print("ESCOLHA RECOMENDADA:".center(60, "="))
print()
print("Para AMBIENTE DE PRODUCAO (Sprint 05):")
print("  -> GPT4All API Server (mesma interface que OpenAI)")
print("     Vantagem: codigo swapping entre local e cloud API")
print()
print("Para AMBIENTE DE DESENVOLVIMENTO (testes rapidos):")
print("  -> GPT4All Direto (sem overhead de servidor)")
print()
print("Para FALLBACK (modelo nao disponivel):")
print("  -> Heuristica de palavras-chave (sempre funciona)")
print()
print("Para PROXIMO NOTEBOOK (c05_pipeline_rag.ipynb):")
print("  -> Usar GPT4All API Server como backend padrao")
print("     (permite trocar para OpenAI remotamente se necessario)")
registro.info("Conclusao: GPT4All API Server como backend padrao")


## 4.6 Conclusao

### Decisoes tecnicas

- **Backend padrao:** GPT4All API Server (localhost:4891)
- **Fallback:** Heuristica de palavras-chave
- **Interface:** OpenAI-compatible -> permite swap para GPT-4
- **Privacidade:** 100%% local para todos os backends

### Vantagens do GPT4All API Server

- Mesma interface que OpenAI API -> codigo portavel
- Sem dados saindo da maquina (localhost)
- Permite fine-tuning futuro sem mudar codigo

### Limitacoes

- Nenhum dos backends foi possível inicializar (modelo .gguf
  nao disponivel neste ambiente) -> usa heuristica como fallback
- Qualquer LLM local depende de hardware: GPU e preferivel


In [ ]:
registro.info("=" * 60)
registro.info("Caderno 04 concluido.")
registro.info("Fim: %s", datetime.now().isoformat())
print("=" * 60)
print("Caderno 04 -- Inferencia Local: CONCLUIDO")
print("Recomendacao: GPT4All API Server como backend padrao")
